# CVS-Act Surgeon Annotation: 30-Video Menu

Copy of `annotate_audit_v11.ipynb` with a surgeon-specific video menu. It loads the 30 selected video clips, tracks completion separately for surgeon IDs 1, 2, and 3, and writes upload-ready assets to one folder.


In [1]:
from pathlib import Path

from cvs_act.annotation_store import load_clip_manifest
from cvs_act.surgeon_annotation_widget import (
    launch_video_menu_interface,
    prepare_annotation_assets,
)

def find_repo_root(start=Path.cwd()):
    cur = Path(start).resolve()
    for candidate in (cur, *cur.parents):
        if (candidate / 'data').is_dir() and (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')

REPO = find_repo_root()
SAMPLE_ROOT = REPO / 'data/processed/CVS_Challenge_SAGES_v1/cvs_act_surgeon_annotations'
SELECTED_VIDEO_CLIPS = SAMPLE_ROOT / 'selected_video_clips.csv'
ANNOTATION_PATH = SAMPLE_ROOT / 'annotations/cvs_act_action_annotations.jsonl'
ASSET_ROOT = SAMPLE_ROOT / 'aws_upload_assets'

if not SELECTED_VIDEO_CLIPS.exists():
    raise FileNotFoundError(f'Missing {SELECTED_VIDEO_CLIPS}. Run notebooks/surgeon_validation/stratified_sample_30.ipynb first.')

selected_clips = load_clip_manifest(SELECTED_VIDEO_CLIPS)
print(f'Loaded {len(selected_clips)} selected video clips from {SELECTED_VIDEO_CLIPS}')
print(f'Videos: {len(set(c.video_name for c in selected_clips))}')
print(f'Annotations save to: {ANNOTATION_PATH}')
print(f'Per-surgeon mirrors save under: {ANNOTATION_PATH.parent / 'by_surgeon'}')
print(f'Assets save under: {ASSET_ROOT}')
selected_clips[:3]


Loaded 30 selected video clips from /mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_surgeon_annotations/selected_video_clips.csv
Videos: 30
Annotations save to: /mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_surgeon_annotations/annotations/cvs_act_action_annotations.jsonl
Per-surgeon mirrors save under: /mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_surgeon_annotations/annotations/by_surgeon
Assets save under: /mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_surgeon_annotations/aws_upload_assets


[ClipManifestRow(clip_id='35ebdf31-e51a-41cb-b455-3afe865d89f1__C3__avg__c_001650_001950', video_name='35ebdf31-e51a-41cb-b455-3afe865d89f1', granularity='coarse', start_frame=1650, end_frame=1950, manifest_source='/mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_surgeon_annotations/selected_video_clips.csv', split=None, criterion='C3', mind_change=None, existing_cvs_labels={'criterion': 'C3', 'keyframes': [{'role': 'start', 'frame': 1650, 'criteria': {'C1': {'votes': 0, 'total': 3, 'raters': [0, 0, 0]}, 'C2': {'votes': 0, 'total': 3, 'raters': [0, 0, 0]}, 'C3': {'votes': 1, 'total': 3, 'raters': [0, 0, 1]}}}, {'role': 'mid', 'frame': 1800, 'criteria': {'C1': {'votes': 0, 'total': 3, 'raters': [0, 0, 0]}, 'C2': {'votes': 0, 'total': 3, 'raters': [0, 0, 0]}, 'C3': {'votes': 2, 'total': 3, 'raters': [1, 0, 1]}}}, {'role': 'end', 'frame': 1950, 'criteria': {'C1': {'votes': 0, 'total': 3, 'raters': [0, 0, 0]}, 'C2': {'votes': 0, 'total': 3, 'raters': [0, 0, 0]}, 'C3':

Run this asset-prep cell when you want to materialize all 30 selected clip videos and frame PNGs into the AWS upload folder. The menu also creates assets lazily as videos are opened.


In [2]:
# Optional but recommended before uploading assets to AWS.
asset_summary = prepare_annotation_assets(selected_clips, assets_root=ASSET_ROOT)
asset_summary


{'clips': 30, 'missing_frames': 0, 'failed_videos': 0}

In [3]:
ui = launch_video_menu_interface(
    SELECTED_VIDEO_CLIPS,
    annotation_path=ANNOTATION_PATH,
    assets_root=ASSET_ROOT,
)
ui


Run this after surgeon labels exist.


In [ ]:
from cvs_act.annotation_agreement import print_agreement
from cvs_act.annotation_store import export_for_analysis

df = export_for_analysis(selected_clips, annotator_ids=['surgeon_1', 'surgeon_2', 'surgeon_3'], annotations_path=ANNOTATION_PATH)
print_agreement(df, annotator_ids=['surgeon_1', 'surgeon_2', 'surgeon_3'], label_field='action_code')
